In [ ]:
'''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA ENGINEERING PREPARATION
DAY 11 EXERCISES
Difficulty: Intermediate to Advanced
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Rule: Write all logic yourself.
No copy-paste from internet or AI.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PYTHON REVISION
Topics: Functions, *args, **kwargs,
        Pandas, map, filter, lambda
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Exercise 1 - Functions revision: real scenario

You have this list of employee records:

employees = [
    {"name": "Alice",  "dept": "Engineering",
     "salary": 85000, "years": 5},
    {"name": "Bob",    "dept": "Marketing",
     "salary": 62000, "years": 3},
    {"name": "Carol",  "dept": "Engineering",
     "salary": 92000, "years": 8},
    {"name": "David",  "dept": "HR",
     "salary": 55000, "years": 2},
    {"name": "Eva",    "dept": "Finance",
     "salary": 78000, "years": 6},
    {"name": "Frank",  "dept": "Engineering",
     "salary": 88000, "years": 7},
    {"name": "Grace",  "dept": "Finance",
     "salary": 67000, "years": 4},
]

Write these functions:

1. get_dept_summary(employees, dept)
   Returns dict with keys:
   count, avg_salary, max_salary, min_salary
   for the given department only.

2. apply_increment(employees, *dept_names,
                   **increment_rates)
   For each employee:
   - If their dept is in dept_names AND
     a rate exists in increment_rates
     for that dept, apply the rate.
   - Otherwise apply a default of 5 percent.
   Returns a new list with updated salaries.
   Do NOT modify the original list.

3. generate_payslip(employee, **options)
   Options: currency (default "Rs."),
            show_tax (default True),
            tax_rate (default 0.10)
   Prints a formatted payslip for one employee.

Test all three and print results cleanly.

Concepts: functions, *args, **kwargs,
          list of dicts, deepcopy or new dict


---------------------------------------------

Exercise 2 - Pandas revision: harder filtering

Create employees.csv from the data above.
Add a column: active (True/False alternating).

Load it into a DataFrame and:

1. Filter employees where salary is above
   the mean salary of their own department.
   Hint: use groupby().transform('mean')
   df[df['salary'] > df.groupby('dept')
      ['salary'].transform('mean')]

2. Add a column called experience_band:
   years >= 7 → "Senior"
   years >= 4 → "Mid"
   below 4    → "Junior"
   Use pd.cut() or apply().

3. Find the department where the gap between
   max and min salary is the largest.
   Use groupby().agg() then subtract.

4. For each department, find the employee
   with the highest salary.
   Use groupby().idxmax() to get the index,
   then use that index to slice the DataFrame.

5. Create a summary DataFrame using groupby:
   dept | headcount | avg_salary | top_earner
   where top_earner is the name of the
   highest paid person in that dept.

Concepts: transform(), cut(), idxmax(),
          groupby().agg(), apply()


---------------------------------------------

Exercise 3 - map, filter, lambda revision

Given:
salaries = [85000, 62000, 92000, 55000,
            78000, 88000, 67000]

names    = ["Alice", "Bob", "Carol", "David",
            "Eva", "Frank", "Grace"]

1. Using map + lambda:
   Create a new list of salaries after
   deducting 10 percent tax from each.

2. Using filter + lambda:
   Keep only salaries above 70000.

3. Using map + lambda:
   Create a list of formatted strings:
   "Alice: Rs.85,000"
   for each name-salary pair.
   Hint: use zip(names, salaries) inside map.

4. Chain all three in one expression:
   - Filter salaries above 60000
   - Apply 5 percent increment
   - Sort descending
   Do it in three lines using
   filter, map, sorted.

5. Using any() and all():
   - Check if ANY salary exceeds 90000.
   - Check if ALL salaries are above 50000.
   - Check if ALL names have length > 3.

Concepts: map, filter, lambda, zip,
          sorted, any, all, f-string


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PYSPARK PRACTICE
Topics: SparkSession, DataFrame creation,
        basic operations, methods, joins
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

IMPORTANT NOTE BEFORE STARTING:
Every PySpark exercise has a Pandas
equivalent shown alongside it.
Read both versions every time.
Understand what is the same and
what is different.

─────────────────────────────────────────────
PYSPARK vs PANDAS — CORE DIFFERENCE
─────────────────────────────────────────────

Pandas:
  Runs on ONE machine.
  Data lives in memory (RAM).
  Good for small to medium data.
  Up to a few GBs.

PySpark:
  Runs across MANY machines (cluster).
  Data is distributed across nodes.
  Good for massive data.
  TBs and PBs of data.

In your career as a data engineer,
Pandas is for local processing and
quick analysis.
PySpark is for production pipelines
on BigQuery, Databricks, AWS EMR,
and GCP Dataproc.

─────────────────────────────────────────────
SETUP — Run this once at the top
─────────────────────────────────────────────

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeAnalytics") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

local[*] means use all CPU cores
on your local machine.
This is how you run PySpark without
a cluster during practice.

---------------------------------------------

PySpark Exercise 1 - Creating DataFrames

METHOD 1: From a list of tuples with schema

from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

data = [
    (1,  "Alice", "Engineering", 85000.0, 5),
    (2,  "Bob",   "Marketing",   62000.0, 3),
    (3,  "Carol", "Engineering", 92000.0, 8),
    (4,  "David", "HR",          55000.0, 2),
    (5,  "Eva",   "Finance",     78000.0, 6),
    (6,  "Frank", "Engineering", 88000.0, 7),
    (7,  "Grace", "Finance",     67000.0, 4),
]

schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("dept",       StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("years",      IntegerType(), True),
])

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

METHOD 2: From a CSV file

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("employees.csv")

df.show()
df.printSchema()

PANDAS EQUIVALENT:
import pandas as pd
df = pd.read_csv("employees.csv")
print(df)
print(df.dtypes)

Task:
1. Create the DataFrame using Method 1.
2. Print schema using printSchema().
3. Show first 3 rows using show(3).
4. Print total row count using count().
5. Print column names using df.columns.

---------------------------------------------

PySpark Exercise 2 - Basic DataFrame methods

Using the employee DataFrame from Exercise 1:

1. Select specific columns:

   df.select("name", "dept", "salary").show()

   PANDAS: df[['name','dept','salary']]

2. Filter rows:

   df.filter(df["salary"] > 70000).show()

   PANDAS: df[df['salary'] > 70000]

   Also try:
   from pyspark.sql.functions import col
   df.filter(col("salary") > 70000).show()

3. Filter with multiple conditions:

   df.filter(
       (col("dept") == "Engineering") &
       (col("salary") > 80000)
   ).show()

   PANDAS:
   df[(df['dept']=='Engineering') &
      (df['salary'] > 80000)]

4. Add a new column using withColumn:

   from pyspark.sql.functions import col, round

   df = df.withColumn(
       "monthly_salary",
       round(col("salary") / 12, 2)
   )
   df.show()

   PANDAS:
   df['monthly_salary'] = round(
       df['salary'] / 12, 2)

5. Rename a column:

   df = df.withColumnRenamed(
       "dept", "department"
   )

   PANDAS: df.rename(
       columns={'dept':'department'})

6. Drop a column:

   df = df.drop("monthly_salary")

   PANDAS: df.drop(
       columns=['monthly_salary'])

7. Sort rows:

   df.orderBy(col("salary").desc()).show()

   PANDAS: df.sort_values(
       'salary', ascending=False)

Task:
Perform all 7 operations in sequence.
Print result after each step using show().

---------------------------------------------

PySpark Exercise 3 - Aggregations and
                     groupBy

from pyspark.sql.functions import (
    count, sum, avg, min, max, round
)

1. Count total employees:

   df.count()

   PANDAS: len(df)

2. Summary statistics:

   df.describe().show()

   PANDAS: df.describe()

3. groupBy department — count:

   df.groupBy("dept") \
     .count() \
     .orderBy("count", ascending=False) \
     .show()

   PANDAS:
   df.groupby('dept').size()

4. groupBy department — multiple aggregations:

   df.groupBy("dept").agg(
       count("emp_id").alias("headcount"),
       round(avg("salary"), 2).alias("avg_salary"),
       max("salary").alias("max_salary"),
       min("salary").alias("min_salary")
   ).show()

   PANDAS:
   df.groupby('dept').agg(
       headcount=('emp_id','count'),
       avg_salary=('salary','mean'),
       max_salary=('salary','max'),
       min_salary=('salary','min')
   )

5. Filter after groupBy using .filter()
   on the aggregated result:

   df.groupBy("dept").agg(
       round(avg("salary"),2).alias("avg_sal")
   ).filter(col("avg_sal") > 70000).show()

   PANDAS:
   result = df.groupby('dept')['salary'] \
              .mean().reset_index()
   result[result['salary'] > 70000]

Task:
Run all 5 aggregations.
For each, write the Pandas version
in a comment next to the PySpark code.

---------------------------------------------

PySpark Exercise 4 - Adding columns
                     with conditions
                     (when / otherwise)

In Pandas you used apply() or np.select().
In PySpark you use when() and otherwise().
This is the PySpark equivalent of
SQL CASE WHEN.

from pyspark.sql.functions import when, col

1. Add salary_band column:

   df = df.withColumn(
       "salary_band",
       when(col("salary") >= 85000, "Senior")
       .when(col("salary") >= 65000, "Mid")
       .otherwise("Junior")
   )
   df.show()

   PANDAS:
   def band(s):
       if s >= 85000: return "Senior"
       if s >= 65000: return "Mid"
       return "Junior"
   df['salary_band'] = df['salary'].apply(band)

   SQL EQUIVALENT:
   CASE
     WHEN salary >= 85000 THEN 'Senior'
     WHEN salary >= 65000 THEN 'Mid'
     ELSE 'Junior'
   END

2. Add experience_band column:

   df = df.withColumn(
       "experience_band",
       when(col("years") >= 7, "Veteran")
       .when(col("years") >= 4, "Experienced")
       .otherwise("Fresher")
   )
   df.show()

3. Combine two conditions in when():

   df = df.withColumn(
       "high_value",
       when(
           (col("salary") > 80000) &
           (col("years") > 5),
           "Yes"
       ).otherwise("No")
   )
   df.show()

Task:
Add all three new columns.
Print the final DataFrame with all columns.
Then select only:
name, dept, salary, salary_band,
experience_band, high_value
and show it.

---------------------------------------------

PySpark Exercise 5 - Joins revision

Create a second DataFrame: departments

dept_data = [
    ("Engineering", 1500000, "New York"),
    ("Marketing",    800000, "Chicago"),
    ("Finance",      600000, "New York"),
    ("HR",           400000, "Dallas"),
    ("Legal",        350000, "New York"),
]

dept_schema = StructType([
    StructField("dept",     StringType(),  True),
    StructField("budget",   IntegerType(), True),
    StructField("location", StringType(),  True),
])

dept_df = spark.createDataFrame(
    dept_data, dept_schema
)

Note: Legal has no employees.
      Operations dept not in dept_df.
      This matters for join results.

Now perform these joins:

1. INNER JOIN
   Only employees who have a matching dept.

   result = df.join(dept_df, on="dept",
                    how="inner")
   result.show()

   PANDAS:
   pd.merge(df, dept_df,
            on='dept', how='inner')

   SQL:
   SELECT * FROM employees e
   INNER JOIN departments d
   ON e.dept = d.dept

2. LEFT JOIN
   All employees + dept info where available.

   result = df.join(dept_df, on="dept",
                    how="left")
   result.show()

3. RIGHT JOIN
   All departments + employee info
   where available.
   Legal should appear with nulls.

   result = df.join(dept_df, on="dept",
                    how="right")
   result.show()

4. FULL OUTER JOIN
   Everything from both sides.

   result = df.join(dept_df, on="dept",
                    how="full")
   result.show()

5. After LEFT JOIN, answer this:
   For each employee show their name,
   salary, department budget, and location.
   Select only these four columns.
   Order by salary descending.

Task:
Run all four joins.
After each join print the row count.
Compare: do the counts make sense?
Write a one-line comment explaining
why each count is what it is.

---------------------------------------------

PySpark Exercise 6 - Writing output

After all transformations, write the
final DataFrame to a CSV file:

df.write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("output/employees_processed")

Note: PySpark writes a FOLDER not a file.
Inside you will see part-00000-*.csv files.
This is because PySpark is designed to
write in parallel across many machines.
Each partition writes its own file.

To write a single file (for small data):

df.coalesce(1) \
  .write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("output/employees_single")

PANDAS EQUIVALENT:
df.to_csv("employees_processed.csv",
          index=False)

Also write as Parquet:

df.write \
  .mode("overwrite") \
  .parquet("output/employees_parquet")

Parquet is the industry standard format
for data engineering pipelines.
Always prefer Parquet over CSV in production.
It is columnar, compressed, and much faster.

Task:
1. Write processed DataFrame to CSV folder.
2. Write to single CSV using coalesce(1).
3. Write to Parquet.
4. Read back the Parquet file and show it.

   df_back = spark.read \
       .parquet("output/employees_parquet")
   df_back.show()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SQL PRACTICE
Topic: Level 2 harder revision
       Multi-concept: GROUP BY, HAVING,
       CASE, String Functions combined
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Use your freesql.com setup.
Tables: employees, department_info,
        orders, products, customers,
        appointments, doctors, patients.

---------------------------------------------

SQL Exercise 1 - CASE + GROUP BY combined

1. Classify each employee into a salary band
   using CASE, then count how many employees
   fall into each band.

   Expected output:
   salary_band | count
   Senior      | 3
   Mid         | 2
   Junior      | 2

   Hint: wrap your CASE in a subquery,
   then GROUP BY salary_band on the outside.

   SELECT salary_band, COUNT(*) AS count
   FROM (
       SELECT name,
           CASE
               WHEN salary >= 85000 THEN 'Senior'
               WHEN salary >= 65000 THEN 'Mid'
               ELSE 'Junior'
           END AS salary_band
       FROM employees
       WHERE salary IS NOT NULL
   )
   GROUP BY salary_band
   ORDER BY count DESC;

2. For each department show:
   - dept name
   - number of Senior employees
   - number of Mid employees
   - number of Junior employees
   (all in one row per department)

   Hint: Use SUM(CASE WHEN ... THEN 1 ELSE 0 END)
   This is called CONDITIONAL AGGREGATION.
   It is used constantly in real reporting.

   SELECT department,
       SUM(CASE WHEN salary >= 85000
           THEN 1 ELSE 0 END) AS senior_count,
       SUM(CASE WHEN salary >= 65000
           AND salary < 85000
           THEN 1 ELSE 0 END) AS mid_count,
       SUM(CASE WHEN salary < 65000
           THEN 1 ELSE 0 END) AS junior_count
   FROM employees
   WHERE salary IS NOT NULL
   GROUP BY department
   ORDER BY department;

   This pattern is called PIVOT using CASE.
   Remember it. You will use it in every job.

---------------------------------------------

SQL Exercise 2 - Multi-table GROUP BY

Using employees and department_info:

1. For each department show:
   - department name
   - department location (from dept_info)
   - department budget (from dept_info)
   - total salary of all employees
   - difference: budget minus total salary
   - status: "Within Budget" or "Over Budget"
     using CASE

   Use implicit join:
   FROM employees e, department_info d
   WHERE e.department = d.department

2. Find departments where the number of
   active employees (is_active = 1) is
   greater than the number of inactive ones.

   Hint: Use conditional aggregation again.
   SUM(CASE WHEN is_active=1 THEN 1 ELSE 0 END)

3. From appointments and doctors:
   Show each doctor's name, specialty,
   total appointments, total fee collected,
   and a workload label:
   total appointments >= 3 → "High"
   total appointments = 2  → "Medium"
   total appointments = 1  → "Low"
   Use implicit join + GROUP BY + CASE.

---------------------------------------------

SQL Exercise 3 - String Functions
                 with aggregation

1. From employees:
   Show department in UPPER case,
   count of employees, avg salary rounded.
   Group by department.

2. From customers:
   Concatenate first_name + ' ' + last_name
   as full_name.
   Show full_name, city, country.
   Filter only customers whose full_name
   has more than 10 characters.
   Use LENGTH(first_name || ' ' || last_name)

3. From doctors:
   Remove the "Dr. " prefix from name.
   Show cleaned name and specialty.
   Use REPLACE or SUBSTR.
   Order alphabetically by cleaned name.

4. From patients:
   Group by the first letter of their city.
   Count patients per starting letter.
   Hint: SUBSTR(city, 1, 1) gives first letter.
   Order by count descending.

---------------------------------------------

SQL Exercise 4 - Hard combined challenge

Write ONE query that produces this output
from the orders table:

status_group | order_count | total_revenue
             | pct_of_total| avg_order_value

Where:
- status_group uses CASE to group:
  "Delivered" and "Shipped" → "Fulfilled"
  "Pending"                 → "Pending"
  "Cancelled"               → "Cancelled"
- pct_of_total is each group's revenue
  as a percentage of grand total revenue
- avg_order_value is total_revenue / oder_countr
- Round all decimals to 2 places

Hint for percentage:
  Use a subquery to get grand total,
  then divide group revenue by it,
  multiply by 100.

  SUM(amount) * 100 /
  (SELECT SUM(amount) FROM orders)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LINUX PRACTICE
Topic: Light revision — one pipeline task
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

You have the employees.csv file from
today's Python exercises.

Perform the following entirely in terminal:

1. Count total employees (exclude header).

2. Extract unique departments.
   Show each department and how many
   employees are in it.
   Use: cut | sort | uniq -c | sort -rn

3. Find the highest salary in the file
   without Python.
   Use: sort -t',' -k4 -rn | head -2

4. Find all employees in Engineering.
   Save to engineering_team.txt.
   Use: grep

5. Using awk, print name and salary
   for employees earning above 70000.
   awk -F',' '$4 > 70000 {print $2, $4}'

6. Create a tar.gz archive of all files
   in your current working folder.
   Verify contents using tar -tvf.

Commands: cut  sort  uniq  grep  awk  tar

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MINI PROJECT — DAY 11
Employee Analytics Pipeline
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Goal:
Build an employee analytics pipeline
using BOTH Pandas and PySpark.
Run the same operations in both.
Compare the results.
This is how real data engineers validate
a PySpark migration from Pandas.

---

DATASET

You will use two files:

employees.csv
emp_id,name,dept,salary,years,city
1,Alice,Engineering,85000,5,New York
2,Bob,Marketing,62000,3,Chicago
3,Carol,Engineering,92000,8,New York
4,David,HR,55000,2,Dallas
5,Eva,Finance,78000,6,New York
6,Frank,Engineering,88000,7,Chicago
7,Grace,Finance,67000,4,Dallas
8,Henry,HR,71000,5,New York
9,Ian,Marketing,58000,2,Chicago
10,Jane,Engineering,95000,9,New York

departments.csv
dept,budget,location
Engineering,1500000,New York
Marketing,800000,Chicago
Finance,600000,New York
HR,400000,Dallas
Legal,350000,New York

---

STEP 1 - Linux Setup

mkdir employee_pipeline
cd employee_pipeline
mkdir raw processed reports

Create both CSV files in raw/.
Verify with: wc -l raw/*.csv

---

STEP 2 - Pandas Pipeline

Create: processed/pandas_analysis.py

Task 1 — Load and profile:
- Read both CSVs.
- Print shape, dtypes, null counts.
- Print describe() for employees.

Task 2 — Clean and enrich employees:
- Strip whitespace from string columns.
- Add salary_band using apply():
  >= 85000 → "Senior"
  >= 65000 → "Mid"
  below    → "Junior"
- Add experience_band using apply():
  years >= 7 → "Veteran"
  years >= 4 → "Experienced"
  below 4    → "Fresher"

Task 3 — Merge with departments:
- Left join employees with departments
  on dept column.
- Keep: name, dept, salary, years, city,
        budget, location, salary_band,
        experience_band.
- Save to processed/enriched_employees.csv

Task 4 — Analysis (save each result):
- Dept summary: headcount, avg_salary,
  total_salary, budget, variance
  (budget minus total_salary).
  Save to reports/dept_summary.csv

- Salary band distribution:
  count per band.
  Save to reports/band_distribution.csv

- Top 3 earners per department.
  Save to reports/top_earners.csv

- City-wise headcount and avg salary.
  Save to reports/city_summary.csv

---

STEP 3 - PySpark Pipeline

Create: processed/spark_analysis.py

Repeat ALL four tasks from Step 2
using PySpark.

Use:
- spark.read.csv() for loading
- withColumn() + when() for new columns
- df.join() for merging
- groupBy().agg() for analysis
- df.write.csv() for saving output

Save PySpark outputs to:
reports/spark_dept_summary.csv
reports/spark_band_distribution.csv
reports/spark_top_earners.csv
reports/spark_city_summary.csv

---

STEP 4 - SQL Verification

Run these four queries on freesql.com.
Create the employees and departments
tables with the same 10-row dataset.

1. Dept summary with budget variance.
2. Salary band distribution using
   conditional aggregation.
3. Top earner per department using
   subquery.
4. City-wise headcount and avg salary.

All four SQL results must match both
the Pandas and PySpark outputs.

---

STEP 5 - Comparison check

Create: reports/comparison.py

Write a function that:
- Loads pandas output CSV
- Loads pyspark output CSV
- Compares both DataFrames using
  df1.equals(df2) or a row-by-row check
- Prints MATCH or MISMATCH for each report

This is called a reconciliation check.
In production pipelines this step
confirms that a PySpark migration
produces identical results to the
original Pandas pipeline.

---

STEP 6 - Archive

tar -czvf employee_pipeline_day11.tar.gz
employee_pipeline/

Verify:
tar -tvf employee_pipeline_day11.tar.gz

---

End Goal:
You have run the exact same analytics
pipeline in three ways — Pandas, PySpark,
and SQL — and verified all three produce
identical results.

This is the core skill of a data engineer:
not just knowing one tool, but knowing
how the same operation looks across
all layers of the stack.

Pandas   → local processing, quick analysis
PySpark  → distributed, production scale
SQL      → querying, reporting, verification

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DAILY TARGETS
60 minutes  -- Solve exercises
20 minutes  -- Review and discuss in group
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''


In [ ]:
# Date: 23-04-2026
# DATA ENGINEERING PREPARATION
# DAY 11 EXERCISES
# Difficulty: Intermediate to Advanced

'''
PYTHON REVISION
Topics: Functions, *args, **kwargs,
        Pandas, map, filter, lambda
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Exercise 1 - Functions revision: real scenario

You have this list of employee records:
Write these functions:
Test all three and print results cleanly.

Concepts: functions, *args, **kwargs,
          list of dicts, deepcopy or new dict
---------------------------------------------
'''

employees = [
    {"name": "Alice",  "dept": "Engineering",
     "salary": 85000, "years": 5},
    {"name": "Bob",    "dept": "Marketing",
     "salary": 62000, "years": 3},
    {"name": "Carol",  "dept": "Engineering",
     "salary": 92000, "years": 8},
    {"name": "David",  "dept": "HR",
     "salary": 55000, "years": 2},
    {"name": "Eva",    "dept": "Finance",
     "salary": 78000, "years": 6},
    {"name": "Frank",  "dept": "Engineering",
     "salary": 88000, "years": 7},
    {"name": "Grace",  "dept": "Finance",
     "salary": 67000, "years": 4},
]

In [ ]:
# 1. get_dept_summary(employees, dept)
#    Returns dict with keys:
#    count, avg_salary, max_salary, min_salary
#    for the given department only.
def get_dept_summary(employees, dept):
    
    filtered = [i for i in employees if i["dept"] == dept]

    salarys = [i["salary"] for i in filtered]
    
    count = len(employees)
    count_dept = len(filtered)
    avg_salary = sum(salarys) / count
    max_salary = max(salarys)
    min_salary = min(salarys)

    return {
        "Total_employee": count,
        "No_of_employee_for_dept": count_dept,
        "Avg_salary": avg_salary,
        "Max_salary": max_salary,
        "Min_salary": min_salary
    }

print(get_dept_summary(employees, "Engineering"))

{'Total_employee': 7, 'No_of_employee_for_dept': 3, 'Avg_salary': 37857.142857142855, 'Max_salary': 92000, 'Min_salary': 85000}


In [ ]:
# 2. apply_increment(employees, *dept_names,
#                    **increment_rates)
#    For each employee:
#    - If their dept is in dept_names AND
#      a rate exists in increment_rates
#      for that dept, apply the rate.
#    - Otherwise apply a default of 5 percent.
#    Returns a new list with updated salaries.
#    Do NOT modify the original list.

def apply_increment(employees, *dept_names, **increment_rates):
    updated_salaries = []

    for i in employees:
        new_emp = i.copy()
        #print(new_emp)
    
        depts = i["dept"]
        #print(depts)

        salarys = i["salary"]
        #print(salarys)

        if depts in dept_names and depts in increment_rates:
            rate = increment_rates[depts]
        else:
            rate = 5

        new_salarys = salarys + (salarys * rate) / 100
        new_emp["salary"] = new_salarys

        updated_salaries.append(new_emp)

    return updated_salaries
result = apply_increment(employees, "Engineering", "HR", Engineering=10, HR=7)
print(result)

[{'name': 'Alice', 'dept': 'Engineering', 'salary': 93500.0, 'years': 5}, {'name': 'Bob', 'dept': 'Marketing', 'salary': 65100.0, 'years': 3}, {'name': 'Carol', 'dept': 'Engineering', 'salary': 101200.0, 'years': 8}, {'name': 'David', 'dept': 'HR', 'salary': 58850.0, 'years': 2}, {'name': 'Eva', 'dept': 'Finance', 'salary': 81900.0, 'years': 6}, {'name': 'Frank', 'dept': 'Engineering', 'salary': 96800.0, 'years': 7}, {'name': 'Grace', 'dept': 'Finance', 'salary': 70350.0, 'years': 4}]


In [ ]:
# 3. generate_payslip(employee, **options)
#    Options: currency (default "Rs."),
#             show_tax (default True),
#             tax_rate (default 0.10)
#    Prints a formatted payslip for one employee.

def generate_payslip(employees, **options):
    name = employees["name"]
    dept = employees["dept"]
    salary = employees["salary"]

    currency = options.get("currency", "Rs.")
    show_tax = options.get("show_tax", True)
    tax_rate = options.get("tax_rate", 0.10)

    print("----------PYSLIP----------")
    print(f"Name       : {name}")
    print(f"Department : {dept}")
    print(f"salary    : {currency}{salary}")

    if show_tax:
        tax = salary * tax_rate
        net_salary = salary - tax

        print(f"Tax : ({tax_rate * 100}%) : {currency}{tax}")
        print(f"Net Salary : {currency}{net_salary}")

    else:
        print("Tax         : Not Applied")
        print(f"Net Salary : {currency}{salary}")

    print("--------------------------")

generate_payslip(
    {"name": "Alice",  "dept": "Engineering", "salary": 85000, "years": 5},
    currency = "Rs.",
    show_tax = True,
    tax_rate = 0.10
)

----------PYSLIP----------
Name       : Alice
Department : Engineering
salary    : Rs.85000
Tax : (10.0%) : Rs.8500.0
Net Salary : Rs.76500.0
--------------------------


In [ ]:
# Exercise 2 - Pandas revision: harder filtering
'''
Create employees.csv from the data above.
Add a column: active (True/False alternating).

Load it into a DataFrame and:
Concepts: transform(), cut(), idxmax(),
          groupby().agg(), apply()


---------------------------------------------
'''

In [6]:
import csv 

employees = [
    {"name": "Alice",  "dept": "Engineering",
     "salary": 85000, "years": 5, "is_active": True},
    {"name": "Bob",    "dept": "Marketing",
     "salary": 62000, "years": 3, "is_active": False},
    {"name": "Carol",  "dept": "Engineering",
     "salary": 92000, "years": 8, "is_active": True},
    {"name": "David",  "dept": "HR",
     "salary": 55000, "years": 2, "is_active": False},
    {"name": "Eva",    "dept": "Finance",
     "salary": 78000, "years": 6, "is_active": True},
    {"name": "Frank",  "dept": "Engineering",
     "salary": 88000, "years": 7, "is_active": False},
    {"name": "Grace",  "dept": "Finance",
     "salary": 67000, "years": 4, "is_active": True},
]

with open("employee.csv", "w", newline="") as file:
    writer = csv.writer(file)

    writer.writerow(["name", "dept", "salary", "years", "is_active"])

    row = [[i["name"], i["dept"], i["salary"], i["years"], i["is_active"]] for i in employees]
    writer.writerows(row)
    

In [1]:
# 1. Filter employees where salary is above
#    the mean salary of their own department.
#    Hint: use groupby().transform('mean')
#    df[df['salary'] > df.groupby('dept')
#       ['salary'].transform('mean')]
import pandas as pd

employee_df = pd.read_csv("employee.csv") 
employee_df.head(7)

employee_df[employee_df["salary"] > employee_df.groupby("dept")["salary"].transform("mean")]

,name,dept,salary,years,is_active
2,Carol,Engineering,92000,8,True
4,Eva,Finance,78000,6,True


In [ ]:
# 2. Add a column called experience_band:
#    years >= 7 → "Senior"
#    years >= 4 → "Mid"
#    below 4    → "Junior"
#    Use pd.cut() or apply().

def get_band(years):
    if years >= 7:
        return "Senior"
    elif years >= 4:
        return "Mid"
    else:
        return "Junior"

employee_df["experience_band"] = employee_df["years"].apply(get_band)
employee_df.head(7)


,name,dept,salary,years,is_active,experience_band
0,Alice,Engineering,85000,5,True,Mid
1,Bob,Marketing,62000,3,False,Junior
2,Carol,Engineering,92000,8,True,Senior
3,David,HR,55000,2,False,Junior
4,Eva,Finance,78000,6,True,Mid
5,Frank,Engineering,88000,7,False,Senior
6,Grace,Finance,67000,4,True,Mid


In [ ]:
# 3. Find the department where the gap between
#    max and min salary is the largest.
#    Use groupby().agg() then subtract.

result = employee_df.groupby("dept")["salary"].agg(["max", "min"])
result["gap"] = result["max"] - result["min"]

max_gap = result["gap"].idxmax()

print(max_gap)

Finance


In [32]:
# 4. For each department, find the employee
#    with the highest salary.
#    Use groupby().idxmax() to get the index,
#    then use that index to slice the DataFrame.

index = employee_df.groupby("dept")["salary"].idxmax()

row = employee_df.loc[index]

print(row)

    name         dept  salary  years  is_active experience_band
2  Carol  Engineering   92000      8       True          Senior
4    Eva      Finance   78000      6       True             Mid
3  David           HR   55000      2      False          Junior
1    Bob    Marketing   62000      3      False          Junior


In [ ]:
# 5. Create a summary DataFrame using groupby:
#    dept | headcount | avg_salary | top_earner
#    where top_earner is the name of the
#    highest paid person in that dept.

summary_df = employee_df.groupby("dept").agg(
    headcount=("name", "count"),
    avg_salary=("salary", "mean")
)
index = employee_df.groupby("dept")["salary"].idxmax()
top_earner = employee_df.loc[index, ["dept", "name"]].set_index("dept")

summary_df["top_earner"] = top_earner["name"]

summary = summary_df.reset_index()
summary.head()

,dept,headcount,avg_salary,top_earner
0,Engineering,3,88333.333333,Carol
1,Finance,2,72500.000000,Eva
2,HR,1,55000.000000,David
3,Marketing,1,62000.000000,Bob


In [ ]:
# Exercise 3 - map, filter, lambda revision
'''
Given:
salaries = [85000, 62000, 92000, 55000,
            78000, 88000, 67000]

names    = ["Alice", "Bob", "Carol", "David",
            "Eva", "Frank", "Grace"]

Concepts: map, filter, lambda, zip,
          sorted, any, all, f-string


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''


In [14]:
# 1. Using map + lambda:
#    Create a new list of salaries after
#    deducting 10 percent tax from each.

salaries = [85000, 62000, 92000, 55000,
            78000, 88000, 67000]

names    = ["Alice", "Bob", "Carol", "David",
            "Eva", "Frank", "Grace"]

deducting_10_percent = list(map(lambda x: x - (x * 0.1), salaries))
print(f"After deducting 10 percent tax salary is: {deducting_10_percent}")

After deducting 10 percent tax salary is: [76500.0, 55800.0, 82800.0, 49500.0, 70200.0, 79200.0, 60300.0]


In [15]:
# 2. Using filter + lambda:
#    Keep only salaries above 70000.

filter_list = list(filter(lambda x: x > 70000, deducting_10_percent))
print(f"Salary Above 70000 list is: {filter_list}")

Salary Above 70000 list is: [76500.0, 82800.0, 70200.0, 79200.0]


In [22]:
# 3. Using map + lambda:
#    Create a list of formatted strings:
#    "Alice: Rs.85,000"
#    for each name-salary pair.
#    Hint: use zip(names, salaries) inside map.

formatted_str = list(map(lambda x: f"{x[0]}: Rs.{x[1]:,}", zip(names, salaries)))
print(formatted_str)

# for name, salary in zip(names, salaries):
#     print(f"{name}: Rs.{salary}")

['Alice: Rs.85,000', 'Bob: Rs.62,000', 'Carol: Rs.92,000', 'David: Rs.55,000', 'Eva: Rs.78,000', 'Frank: Rs.88,000', 'Grace: Rs.67,000']


In [32]:
# 4. Chain all three in one expression:
#    - Filter salaries above 60000
#    - Apply 5 percent increment
#    - Sort descending
#    Do it in three lines using
#    filter, map, sorted.

filter_list = list(filter(lambda x: x > 60000, salaries))
#print(filter_list)
updated_salary = list(map(lambda x: x + (x * 0.05), filter_list))
#print(updated_salary)

desc = sorted(updated_salary, reverse=True)
print(desc)

[96600.0, 92400.0, 89250.0, 81900.0, 70350.0, 65100.0]


In [39]:
# 5. Using any() and all():
#    - Check if ANY salary exceeds 90000.
#    - Check if ALL salaries are above 50000.
#    - Check if ALL names have length > 3.

High = any(i > 90000 for i in salaries)
print(High)

all_salary_above_50k = all(s > 50000 for s in salaries)
print(all_salary_above_50k)

name_len_above_3 = all(len(name) > 3 for name in names)
print(name_len_above_3)

True
True
False


In [ ]:
# SQL PRACTICE
'''
Topic: Level 2 harder revision
       Multi-concept: GROUP BY, HAVING,
       CASE, String Functions combined
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Use your freesql.com setup.
Tables: employees, department_info,
        orders, products, customers,
        appointments, doctors, patients.

---------------------------------------------

SQL Exercise 1 - CASE + GROUP BY combined

1. Classify each employee into a salary band
   using CASE, then count how many employees
   fall into each band.

   Expected output:
   salary_band | count
   Senior      | 3
   Mid         | 2
   Junior      | 2

   Hint: wrap your CASE in a subquery,
   then GROUP BY salary_band on the outside.

   SELECT salary_band, COUNT(*) AS count
   FROM (
       SELECT name,
           CASE
               WHEN salary >= 85000 THEN 'Senior'
               WHEN salary >= 65000 THEN 'Mid'
               ELSE 'Junior'
           END AS salary_band
       FROM employees
       WHERE salary IS NOT NULL
   )
   GROUP BY salary_band
   ORDER BY count DESC;

2. For each department show:
   - dept name
   - number of Senior employees
   - number of Mid employees
   - number of Junior employees
   (all in one row per department)

   Hint: Use SUM(CASE WHEN ... THEN 1 ELSE 0 END)
   This is called CONDITIONAL AGGREGATION.
   It is used constantly in real reporting.

   SELECT department,
       SUM(CASE WHEN salary >= 85000
           THEN 1 ELSE 0 END) AS senior_count,
       SUM(CASE WHEN salary >= 65000
           AND salary < 85000
           THEN 1 ELSE 0 END) AS mid_count,
       SUM(CASE WHEN salary < 65000
           THEN 1 ELSE 0 END) AS junior_count
   FROM employees
   WHERE salary IS NOT NULL
   GROUP BY department
   ORDER BY department;

   This pattern is called PIVOT using CASE.
   Remember it. You will use it in every job.

---------------------------------------------
'''
# 1
SELECT * FROM EMPLOYEES;

SELECT SALARY_BAND, COUNT(*) AS COUNT
FROM (
    SELECT NAME,
    CASE
        WHEN SALARY >= 85000 THEN 'Senior'
        WHEN SALARY >= 65000 THEN 'Mid'
        ELSE 'Junior'
    END AS SALARY_BAND
    FROM EMPLOYEES
    WHERE SALARY IS NOT NULL
)
GROUP BY SALARY_BAND
ORDER BY COUNT DESC;

# 2
SELECT DEPARTMENT,
    SUM(CASE WHEN SALARY >= 85000
        THEN 1 ELSE 0 END) AS SENIOR_COUNT,
    SUM(CASE WHEN SALARY >= 65000 AND SALARY < 85000
        THEN 1 ELSE 0 END) AS MID_COUNT,
    SUM(CASE WHEN SALARY < 65000
        THEN 1 ELSE 0 END) AS JUNIOR_COUNT
FROM EMPLOYEES
WHERE DEPARTMENT IS NOT NULL
GROUP BY DEPARTMENT
ORDER BY DEPARTMENT;

SELECT * FROM EMPLOYEES;

In [ ]:
# SQL Exercise 2 - Multi-table GROUP BY
'''
Using employees and department_info:

1. For each department show:
   - department name
   - department location (from dept_info)
   - department budget (from dept_info)
   - total salary of all employees
   - difference: budget minus total salary
   - status: "Within Budget" or "Over Budget"
     using CASE

   Use implicit join:
   FROM employees e, department_info d
   WHERE e.department = d.department

2. Find departments where the number of
   active employees (is_active = 1) is
   greater than the number of inactive ones.

   Hint: Use conditional aggregation again.
   SUM(CASE WHEN is_active=1 THEN 1 ELSE 0 END)

3. From appointments and doctors:
   Show each doctor's name, specialty,
   total appointments, total fee collected,
   and a workload label:
   total appointments >= 3 → "High"
   total appointments = 2  → "Medium"
   total appointments = 1  → "Low"
   Use implicit join + GROUP BY + CASE.

---------------------------------------------
'''
# 1
SELECT 
    DI.DEPT_NAME, 
    DI.LOCATION, 
    DI.BUDGET, 
    SUM(DOC.SALARY) AS TOTAL_SALARY, 
    (DI.BUDGET - SUM(DOC.SALARY)) AS DIFFERENCE,

    CASE
        WHEN SUM(DOC.SALARY) <= DI.BUDGET THEN 'Within Budget'
        ELSE 'Over Budget'
    END AS STATUS 

FROM DOCTORS DOC, DEPARTMENT_INFO DI
WHERE DOC.DEPT_ID = DI.DEPT_ID
GROUP BY DI.DEPT_NAME, DI.LOCATION, DI.BUDGET;

# 2

SELECT DEPARTMENT,
    SUM(CASE WHEN IS_ACTIVE = 1 THEN 1 ELSE 0 END) AS ACTIVE_EMP_COUNT
FROM EMPLOYEES
WHERE DEPARTMENT IS NOT NULL
GROUP BY DEPARTMENT
ORDER BY ACTIVE_EMP_COUNT DESC;

# 3
SELECT 
    DOC.DOCTOR_NAME,
    DOC.SPECIALIZATION,
    COUNT(AP.APPOINTMENT_ID) AS TOTAL_APPOINTMENT,
    SUM(AP.FEE) AS TOTAL_FEE_COLL,
    CASE
        WHEN COUNT(AP.APPOINTMENT_ID) >= 3 THEN 'High'
        WHEN COUNT(AP.APPOINTMENT_ID) = 2 THEN 'Medium'
        WHEN COUNT(AP.APPOINTMENT_ID) = 1 THEN 'Low'
        ELSE 'No Work'
    END AS WORKLOAD
FROM DOCTORS DOC, APPOINTMENTS AP
WHERE DOC.DOCTOR_ID = AP.DOCTOR_ID

GROUP BY DOC.DOCTOR_NAME, DOC.SPECIALIZATION;

In [ ]:
# SQL Exercise 3 - String Functions with aggregation
'''
1. From employees:
   Show department in UPPER case,
   count of employees, avg salary rounded.
   Group by department.

2. From customers:
   Concatenate first_name + ' ' + last_name
   as full_name.
   Show full_name, city, country.
   Filter only customers whose full_name
   has more than 10 characters.
   Use LENGTH(first_name || ' ' || last_name)

3. From doctors:
   Remove the "Dr. " prefix from name.
   Show cleaned name and specialty.
   Use REPLACE or SUBSTR.
   Order alphabetically by cleaned name.

4. From patients:
   Group by the first letter of their city.
   Count patients per starting letter.
   Hint: SUBSTR(city, 1, 1) gives first letter.
   Order by count descending.

---------------------------------------------
'''
# 1
SELECT
    UPPER(DEPARTMENT) AS UPPER_CASE,
    COUNT(*) AS EMP_COUNT,
    ROUND(AVG(SALARY), 2) AS AVG_SALARY
FROM EMPLOYEES
WHERE DEPARTMENT IS NOT NULL
GROUP BY DEPARTMENT;

# 2
SELECT 
    FIRST_NAME || ' ' || LAST_NAME AS FULL_NAME,
    CITY,
    COUNTRY
FROM CUSTOMERS1
WHERE LENGTH(FIRST_NAME || ' ' || LAST_NAME) > 10;

# 3
SELECT 
    REPLACE(DOCTOR_NAME, 'Dr. ', '') AS CLEANED_NAME,
    SPECIALIZATION
FROM DOCTORS
ORDER BY CLEANED_NAME;

SELECT 
    REPLACE(DOCTOR_NAME, 'Dr. ', 'Mr.') AS CLEANED_NAME,
    SPECIALIZATION
FROM DOCTORS
ORDER BY CLEANED_NAME;

SELECT 
    SUBSTR(DOCTOR_NAME, 5) AS CLEANED_NAME,
    SPECIALIZATION
FROM DOCTORS
ORDER BY CLEANED_NAME;

# 4
SELECT
    SUBSTR(CITY, 1,1) AS FIRST_LETTER,
    COUNT(*) AS PATIENTS_COUNT
FROM PATIENTS
GROUP BY SUBSTR(CITY, 1,1);
ORDER BY PATIENTS_COUNT DESC;


ALTER TABLE patients ADD city VARCHAR2(100);
UPDATE patients SET city = 'Mumbai' WHERE patient_id = 1;
UPDATE patients SET city = 'Pune' WHERE patient_id = 2;
UPDATE patients SET city = 'Delhi' WHERE patient_id = 3;
UPDATE patients SET city = 'Nagpur' WHERE patient_id = 4;
UPDATE patients SET city = 'Chennai' WHERE patient_id = 5;
UPDATE patients SET city = 'Mumbai' WHERE patient_id = 6;
UPDATE patients SET city = 'Pune' WHERE patient_id = 7;
UPDATE patients SET city = 'Delhi' WHERE patient_id = 8;
UPDATE patients SET city = 'Nagpur' WHERE patient_id = 9;
UPDATE patients SET city = 'Chennai' WHERE patient_id = 10;

In [ ]:
'''
SQL Exercise 4 - Hard combined challenge

Write ONE query that produces this output
from the orders table:

status_group | order_count | total_revenue
             | pct_of_total| avg_order_value

Where:
- status_group uses CASE to group:
  "Delivered" and "Shipped" → "Fulfilled"
  "Pending"                 → "Pending"
  "Cancelled"               → "Cancelled"
- pct_of_total is each group's revenue
  as a percentage of grand total revenue
- avg_order_value is total_revenue / oder_countr
- Round all decimals to 2 places

Hint for percentage:
  Use a subquery to get grand total,
  then divide group revenue by it,
  multiply by 100.

  SUM(amount) * 100 /
  (SELECT SUM(amount) FROM orders)
'''
SELECT
    STATUS_GROUP,
    ORDER_COUNT,
    TOTAL_REVENUE,

    ROUND(
        (TOTAL_REVENUE * 100) /
        (SELECT SUM(ORDER_TOTAL) FROM ORDERS), 
     2) AS PCT_OF_TOTAL,

    ROUND(
        TOTAL_REVENUE / ORDER_COUNT,
    2) AS AVG_ORDER_VALUE

    FROM(
        SELECT 
            CASE
                WHEN ORDER_STATUS IN ('Delivered', 'Shipped') THEN 'Fulfilled'
                WHEN ORDER_STATUS = 'Pending' THEN 'Pending'
                WHEN ORDER_STATUS = 'Cancelled' THEN 'Cancelled'
                END AS   STATUS_GROUP,

                COUNT(*) AS ORDER_COUNT,
                SUM(ORDER_TOTAL) AS TOTAL_REVENUE

        FROM ORDERS
        GROUP BY 
            CASE 
                WHEN ORDER_STATUS IN ('Delivered', 'Shipped') THEN 'Fulfilled'
                WHEN ORDER_STATUS = 'Pending' THEN 'Pending'
                WHEN ORDER_STATUS = 'Cancelled' THEN 'Cancelled'
            END
    );

SELECT * FROM ORDERS;

In [ ]:
# PYSPARK PRACTICE - Answer is 'google colab' platform
'''
Topics: SparkSession, DataFrame creation,
        basic operations, methods, joins
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

IMPORTANT NOTE BEFORE STARTING:
Every PySpark exercise has a Pandas
equivalent shown alongside it.
Read both versions every time.
Understand what is the same and
what is different.

─────────────────────────────────────────────
PYSPARK vs PANDAS — CORE DIFFERENCE
─────────────────────────────────────────────

Pandas:
    Runs on ONE machine.
    Data lives in memory (RAM).
    Good for small to medium data.
    Up to a few GBs.

PySpark:
    Runs across MANY machines (cluster).
    Data is distributed across nodes.
    Good for massive data.
    TBs and PBs of data.

In your career as a data engineer,
Pandas is for local processing and
quick analysis.

PySpark is for production pipelines
on BigQuery, Databricks, AWS EMR,
and GCP Dataproc.

─────────────────────────────────────────────
SETUP — Run this once at the top
─────────────────────────────────────────────

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeAnalytics") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

local[*] means use all CPU cores
on your local machine.
This is how you run PySpark without
a cluster during practice.

---------------------------------------------
'''

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LINUX PRACTICE
# Topic: Light revision — one pipeline task
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

'''
You have the employees.csv file from
today's Python exercises.

Perform the following entirely in terminal:

1. Count total employees (exclude header).

   command :  tail -n +2 employee.csv | wc -l

2. Extract unique departments.
   Show each department and how many
   employees are in it.
   Use: cut | sort | uniq -c | sort -rn
   
   command :  cut -d',' -f2 employee.csv | tail -n +2 | sort | uniq -c | sort -rn
   
   Explanation
      cut -d',' -f3 → extracts department column
      tail -n +2 → removes header
      sort → sorts department names
      uniq -c → counts occurrences
      sort -rn → sorts counts in descending order

3. Find the highest salary in the file
   without Python.
   Use: sort -t',' -k4 -rn | head -2

   command1 : sort -t',' -k4 -rn employee.csv | head -2
   command2 : head -1 employee.csv && tail -n +2 employee.csv | sort -t',' -k4 -rn | head -1

4. Find all employees in Engineering.
   Save to engineering_team.txt.
   Use: grep

   command :  grep "Engineering" employee.csv > engineering_team.txt

5. Using awk, print name and salary
   for employees earning above 70000.
   awk -F',' '$4 > 70000 {print $2, $4}'

   command : awk -F',' '$3 > 70000 {print $1, $3}' employee.csv

6. Create a tar.gz archive of all files
   in your current working folder.
   Verify contents using tar -tvf.

Commands: cut  sort  uniq  grep  awk  tar
'''